# YOLOv5 Weed Detection Training
This notebook handles the dataset preparation and training for **YOLOv5**.

### ⚠️ Cache Fix Included
This version automatically cleans up corrupted `.cache` files to prevent loading errors on Windows.

In [1]:
import os
import sys
import json
import shutil
import random
import subprocess
from pathlib import Path
from datetime import datetime
import pandas as pd

# Settings
DATASET_ROOT = r"c:\Users\ahmad\Desktop\computer vision\cottonweed"
WORK_DIR = r"c:\Users\ahmad\Desktop\computer vision\weed_yolo_benchmark"
FRACTION = 0.10  
EPOCHS = 50
BATCH_SIZE = 8
IMG_SIZE = 640
DEVICE = 0

CLASS_NAMES = ["weed1", "weed2", "weed3", "weed4", "weed5", "weed6", "weed7", "weed8", "weed9", "weed10", "weed11", "weed12"]

os.makedirs(WORK_DIR, exist_ok=True)

### 1. Prepare Dataset Subset

In [2]:
def create_subset(fraction):
    subset_name = f"subset_{int(fraction * 100)}"
    subset_root = Path(WORK_DIR) / "datasets" / subset_name
    
    if (subset_root / "SUBSET_READY.json").exists():
        return subset_root

    print(f"Creating {subset_name} subset...")
    os.makedirs(subset_root / "images" / "train", exist_ok=True)
    os.makedirs(subset_root / "labels" / "train", exist_ok=True)
    
    for split in ["val", "test"]:
        src_img = Path(DATASET_ROOT) / "images" / split
        if src_img.exists():
            shutil.copytree(src_img, subset_root / "images" / split, dirs_exist_ok=True)
            shutil.copytree(Path(DATASET_ROOT) / "labels" / split, subset_root / "labels" / split, dirs_exist_ok=True)

    train_images = list((Path(DATASET_ROOT) / "images" / "train").glob("*.jpg"))
    random.seed(42)
    selected = random.sample(train_images, int(len(train_images) * fraction))
    
    for img_path in selected:
        shutil.copy2(img_path, subset_root / "images" / "train")
        lbl_path = Path(DATASET_ROOT) / "labels" / "train" / f"{img_path.stem}.txt"
        if lbl_path.exists():
            shutil.copy2(lbl_path, subset_root / "labels" / "train")

    yaml_content = f"train: { (subset_root / 'images' / 'train').as_posix() }\nval: { (subset_root / 'images' / 'val').as_posix() }\nnc: {len(CLASS_NAMES)}\nnames: {CLASS_NAMES}"
    with open(subset_root / "data.yaml", "w") as f:
        f.write(yaml_content)
        
    with open(subset_root / "SUBSET_READY.json", "w") as f:
        json.dump({"fraction": fraction}, f)
    
    return subset_root

subset_path = create_subset(FRACTION)

### 2. Setup YOLOv5 Repo

In [3]:
repo_dir = Path(WORK_DIR) / "repos" / "yolov5"
if not repo_dir.exists():
    !git clone https://github.com/ultralytics/yolov5 "{repo_dir}"
    !pip install -r "{repo_dir}/requirements.txt"
else:
    print("YOLOv5 repo already exists.")

YOLOv5 repo already exists.


### 3. Training (Real-time output)

In [4]:
exp_name = f"yolov5_data{int(FRACTION*100)}_aug"
exp_dir = Path(WORK_DIR) / "runs" / exp_name
data_yaml = subset_path / "data.yaml"

def clear_cache(path):
    """Force delete .cache files to prevent loading errors on Windows."""
    print("Cleaning up dataset cache files...")
    for cache in path.rglob("*.cache"):
        try:
            os.remove(cache)
            print(f"  - Removed {cache.name}")
        except Exception as e:
            print(f"  - Warning: Could not remove {cache.name}: {e}")

# Check for resume
last_ckpt = exp_dir / "weights" / "last.pt"
best_ckpt = exp_dir / "weights" / "best.pt"

if best_ckpt.exists():
    print("Training already finished. Skipping.")
else:
    # Clear cache to avoid corrupted loading
    clear_cache(subset_path)
    
    # Build the command list
    cmd = [
        sys.executable, str(repo_dir / "train.py"),
        "--img", str(IMG_SIZE),
        "--batch", str(BATCH_SIZE),
        "--epochs", str(EPOCHS),
        "--data", str(data_yaml),
        "--project", f"{WORK_DIR}/runs",
        "--name", exp_name,
        "--exist-ok",
        "--device", str(DEVICE),
        "--workers", "0"
    ]
    
    if last_ckpt.exists():
        cmd.extend(["--resume", str(last_ckpt)])
    else:
        cmd.extend(["--weights", "yolov5s.pt"])

    print(f"Starting training: {' '.join(cmd)}\n")
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

Training already finished. Skipping.


### 4. Validation & Results

In [5]:
val_cmd = [
    sys.executable, str(repo_dir / "val.py"),
    "--weights", str(exp_dir / "weights" / "best.pt"),
    "--data", str(data_yaml),
    "--img", str(IMG_SIZE)
]
print(f"Starting validation: {' '.join(val_cmd)}\n")
process = subprocess.Popen(val_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
process.wait()

# Save simple marker for comparison
results_csv = exp_dir / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    last = df.iloc[-1]
    summary = {
        "model": "yolov5",
        "fraction": FRACTION,
        "map50": float(last.get('     metrics/mAP_0.5', 0)),
                "precision": float(last.get('   metrics/precision', 0)),
                "recall": float(last.get('      metrics/recall', 0))
    }
    with open(exp_dir / "EXPERIMENT_DONE.json", "w") as f:
        json.dump(summary, f)

Starting validation: e:\miniconda3\envs\whisper\python.exe c:\Users\ahmad\Desktop\computer vision\weed_yolo_benchmark\repos\yolov5\val.py --weights c:\Users\ahmad\Desktop\computer vision\weed_yolo_benchmark\runs\yolov5_data10_aug\weights\best.pt --data c:\Users\ahmad\Desktop\computer vision\weed_yolo_benchmark\datasets\subset_10\data.yaml --img 640

val: data=c:\Users\ahmad\Desktop\computer vision\weed_yolo_benchmark\datasets\subset_10\data.yaml, weights=['c:\\Users\\ahmad\\Desktop\\computer vision\\weed_yolo_benchmark\\runs\\yolov5_data10_aug\\weights\\best.pt'], batch_size=32, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=val, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=weed_yolo_benchmark\repos\yolov5\runs\val, name=exp, exist_ok=False, half=False, dnn=False
fatal: cannot change to 'C:\Users\ahmad\Desktop\computer': No such file or directory
YOLOv5  2026-5-9 Python-3.